# NB-19 · Jupyter companion

Calculations use the private Workbench API, pinned to version **1.43.0**. You need your individual API key and the private input bundle supplied for **NB-19**.

1. Open this notebook in a fresh Python 3.12 or 3.13 kernel. Keep `notebook-methods-v1.43.0.zip` beside it when working locally; otherwise setup downloads the checksum-verified public helper archive.
2. Run the helper setup cell, then select your private input directory or ZIP. Local Jupyter can use `EWB_NOTEBOOK_INPUT_DIR` or `EWB_NOTEBOOK_INPUT_ZIP`. Colab uploads send inputs to your Google runtime; use local Jupyter when the inputs must stay on your computer.
3. Run the teaching cells in order. Inspect inputs, receipts, tables and plots locally. Results are saved beneath `data/processed/notebook-results`; save the notebook privately after execution.

The download contains no saved execution outputs. [Previously reviewed figures and results](https://sherandoresearch.github.io/power-markets-series/NB-19.html) remain available in the static reader. A successful calculation validates its numerical contract, not a scientific claim. Saved results retain their original version; these notebooks request version 1.43.0 explicitly.

The static reader preserves previously reviewed results; rerunning this notebook creates new private execution evidence.

<!-- book-volume-context -->
## Where this fits in the volumes

This tests what survives aggregation: an equivalent boundary response need not preserve internal flows, settlement targets, or all path exposures.

- [Volume II, Chapter 4 — Green’s functions and the resistance metric](https://sherandoresearch.github.io/power-markets-series/vol2_grid_geometry.html#c4)
- [Volume V, §4.7 — Aggregation and loss of information](https://sherandoresearch.github.io/power-markets-series/vol5_representing_power_markets.html#c4-7)

In [ ]:
# Install the reviewed transport and notebook helpers. This cell makes no API calls.
import hashlib, importlib.metadata, io, json, os, pathlib, ssl, stat, subprocess, sys, tempfile, urllib.request, zipfile

if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Use a Python 3.12 or 3.13 kernel for this reviewed release.")
helper_name = "notebook-methods-v1.43.0.zip"
expected_sha256 = "2bea8f592cae108b96369caded46f15631a60f1cd22d033cdd46905c3ce0ee18"
expected_files = json.loads('{"requirements.txt": "d3057422eeabe167fdc242f31c5038add6e0aa9aa73fb31227eb92b688cbb553", "src/ewb_methods_client/__init__.py": "faddf128f002243598cb56ab227945e50a2c6219453f24a2c633bf32c7ccf4ab", "src/methods_notebooks/__init__.py": "94e70a66bebea6221748a48192079f14297c552697aebc78205fd49128f9a345", "src/methods_notebooks/session.py": "86f788c33cc0db110ae1ae41f5cd4588af7029d8cd3aa935e491cabf6af40b3e", "src/methods_notebooks/setup.py": "17990abbd57a1472a44071834a7364604b452694605e6e4b19624ef8377e6d9f"}')
helper_path = pathlib.Path(os.environ.get("EWB_NOTEBOOK_HELPERS_ZIP", helper_name)).expanduser()
if helper_path.is_file():
    helper_bytes = helper_path.read_bytes()
else:
    helper_url = "https://github.com/sherandoresearch/power-markets-notebooks/releases/download/notebooks-v1.43.0/" + helper_name
    # Only the public, checksum-pinned transport helpers are downloaded here.
    # Inputs and API credentials are selected in the next cell.
    # Python.org macOS installs may lack a system CA bundle. Use the verified
    # certifi trust store when present; certificate and hostname checks stay on.
    try:
        import certifi
    except ImportError:
        tls_context = ssl.create_default_context()
    else:
        tls_context = ssl.create_default_context(cafile=certifi.where())
    with urllib.request.urlopen(helper_url, timeout=90, context=tls_context) as response:
        helper_bytes = response.read(1_000_001)
if len(helper_bytes) > 1_000_000 or hashlib.sha256(helper_bytes).hexdigest() != expected_sha256:
    raise ValueError("Notebook helper release checksum mismatch")
with zipfile.ZipFile(io.BytesIO(helper_bytes)) as helper_zip:
    members = helper_zip.infolist()
    if len(members) != len(expected_files) or {m.filename for m in members} != set(expected_files):
        raise ValueError("Unexpected helper archive contents")
    if any(stat.S_IFMT(m.external_attr >> 16) not in (0, stat.S_IFREG) or
           m.flag_bits & 1 or m.file_size > 100_000 for m in members):
        raise ValueError("Unsupported helper archive member")
    helper_contents = {name: helper_zip.read(name) for name in expected_files}
if any(hashlib.sha256(value).hexdigest() != expected_files[name] for name, value in helper_contents.items()):
    raise ValueError("Notebook helper member checksum mismatch")
requirements = helper_contents["requirements.txt"].decode().splitlines()
missing = []
for requirement in requirements:
    package, version = requirement.split("==")
    try:
        current_version = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        current_version = None
    if current_version != version:
        if package in sys.modules:
            raise RuntimeError("Restart in a fresh kernel before installing " + requirement)
        missing.append(requirement)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
for requirement in requirements:
    package, version = requirement.split("==")
    if importlib.metadata.version(package) != version:
        raise RuntimeError("Dependency version mismatch: " + package)
if any(name == "methods_notebooks" or name.startswith("methods_notebooks.") or
       name == "ewb_methods_client" or name.startswith("ewb_methods_client.") for name in sys.modules):
    raise RuntimeError("Restart the kernel before rerunning helper setup.")
helper_parent = pathlib.Path.cwd() / "data/processed/notebook-helpers"
helper_parent.mkdir(parents=True, exist_ok=True, mode=0o700)
helper_root = pathlib.Path(tempfile.mkdtemp(prefix="v1.43.0-", dir=helper_parent))
for name, value in helper_contents.items():
    target = helper_root / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(value)
sys.path.insert(0, str(helper_root / "src"))
del helper_bytes, helper_contents
print("Notebook helpers verified. Calculations use Workbench 1.43; inputs and results stay private.")


In [ ]:
from methods_notebooks.setup import configure_notebook
setup_status = configure_notebook('NB-19', require_key=True)
print(setup_status)


# NB-19 — Aggregation, Kron reduction and target loss
Companion to Volume III §4, Volume IV §5 and Volume V §13.7.
Prerequisites: NB-04 and NB-15. Reuse Canonical E and the 18 saved balanced
path vectors H; these are existing unit-transfer columns, not newly generated
random injections. The original six retained buses are 1,7,9,10,12,22.

Electrical elimination preserves a boundary response under stated assumptions.
Regional averaging of observations and a statistical count-network model have
different targets. None reconstructs real PJM topology from market prices.

In [ ]:
NOTEBOOK_ID='NB-19'

import copy, json
from datetime import datetime
from types import SimpleNamespace
import numpy as np
from methods_notebooks.session import Session, MethodsError, setup_plot
session = Session(NOTEBOOK_ID, version='1.43.0')
inputs = session.inputs
responses, execution_plan = {}, []
def execute(name, method, request, parents=()):
    request = copy.deepcopy(request)
    if parents:
        request['parent_runs'] = [{k:responses[p]['receipt'][k] for k in
            ('run_id','scientific_method_id','method_version','result_sha256')} for p in parents]
    response = session.execute(method, request)
    responses[name] = response
    execution_plan.append(dict(name=name,method=method,request=request,parents=list(parents),rest_receipt=response['receipt']))
    (session.output/'execution-plan.json').write_text(json.dumps(execution_plan,indent=2,allow_nan=False)+'\n')
    return {k:response[k] for k in ('result','diagnostics')}
def reject(method, request):
    try:
        session.execute(method, request)
    except MethodsError as exc:
        assert str(exc) == 'Methods request refused (HTTP 422)'
        print('Expected remote rejection:',method,str(exc))
    else:
        raise AssertionError('Invalid retained-input mutation accepted')
plt=setup_plot()

from matplotlib.colors import LinearSegmentedColormap
nord_map=LinearSegmentedColormap.from_list('nord_diverging',['#5E81AC','#ECEFF4','#BF616A'])
print('Local original-path inspection:',np.asarray(inputs['path_matrix'])[:3,:3])
print('Operator provenance:',inputs['network_recovery'])
rows=[];detail={}
for size,request in inputs['requests'].items():
    output=execute('retained_'+size,'kron_response',request)
    result=output['result'];diagnostic=output['diagnostics']['original_network_reconstruction']
    assert diagnostic['original_network_required'] and not diagnostic['reduced_market_equivalence']
    assert not diagnostic['omitted_source_projection_is_physical_solution']
    rows.append(diagnostic['checks'])
    if size=='6':
        detail=dict(r=np.asarray(result['reduced_laplacian']['node_ids'],int),
            e=np.asarray(result['boundary_response_check']['eliminated_node_ids'],int),
            reduced=np.asarray(result['reduced_laplacian']['values']),
            rhs=np.asarray(result['equivalent_injections']['values']),
            original_rhs=np.asarray(diagnostic['original_retained_injections']),
            flow=np.asarray(diagnostic['reconstructed_branch_flows']),
            trueflow=np.asarray(diagnostic['unreduced_branch_flows']))
arrays=dict(H=np.asarray(inputs['path_matrix']),T_intact=np.asarray(inputs['original_intact_exposures']))
research=dict(multiscale=inputs['retained_summaries']);grid=SimpleNamespace(E=inputs['original_constraint_count'])
print('Existing balanced path matrix H:',arrays['H'].shape,'column sums:',arrays['H'].sum(axis=0))
print('Retained buses:',detail['r'].tolist(),'eliminated buses:',detail['e'].tolist())
print('Model: connected lossless DC graph; L=AᵀWA; positive line weights; arbitrary common angle reference.')

## The source term belongs in the Schur complement
Partition buses into retained R and eliminated E. With L_EE invertible,
L_red = L_RR − L_RE L_EE⁻¹L_ER and
p_eff = p_R − L_RE L_EE⁻¹p_E. Solve L_red θ_R = p_eff up to a constant.
Then θ_E = L_EE⁻¹(p_E − L_ER θ_R).

H is sink-minus-source: every existing column sums to zero. Some path endpoints
are eliminated. Keeping just p_R discards their equivalent sources and can even
leave an unbalanced reduced right-hand side. A pseudoinverse then silently
projects away that imbalance; it does not repair the model.

In [ ]:
print('Corrected source balance residual:',rows[0]['source_balance_residual'])
print('Omitted-source imbalance per original path:',detail['original_rhs'].sum(axis=0))
fig,axes=plt.subplots(1,2,figsize=(12,4))
# Draw the graph from its canonical incidence carrier, without regenerating a graph.
rset=set(detail['r']);spec=inputs['diagram']
xy=np.array([[b['x'],b['y']] for b in spec['buses']])
for line in spec['lines']:
    i,j=line['from'],line['to'];axes[0].plot(xy[[i,j],0],xy[[i,j],1],color='#4C566A')
for i,(x,y) in enumerate(xy):
    axes[0].scatter(x,y,color='#88C0D0' if i in rset else '#D08770',s=45);axes[0].text(x+.04,y,str(i),fontsize=7)
axes[0].set(title='Existing retained buses (blue) and eliminated buses',xlabel='Declared diagram x',ylabel='Declared diagram y')
im=axes[1].imshow(detail['reduced'],cmap=nord_map);fig.colorbar(im,ax=axes[1])
axes[1].set(xticks=range(6),xticklabels=detail['r'],yticks=range(6),yticklabels=detail['r'],title='Kron boundary Laplacian',xlabel='Retained bus',ylabel='Retained bus');fig.tight_layout();plt.show()

## Check terminals, reconstructed flows and payout separately
Compare angle differences to the first retained bus, never raw gauge-dependent
angles. Reconstruct all eliminated angles using their known source terms before
computing branch flows WAθ. Under the book's settlement sign convention T=−PSI H,
the unit-transfer flow matrix is −T. Retaining the full constraint dual vector
then reproduces μᵀT for intact hours.

This payout check uses the original full-grid duals; Kron reduction does not
supply them. Dropping internal thermal limits can change dispatch and prices.
Original constraints must be imposed on reconstructed affine branch flows,
with the original limits and eliminated injections. A dense boundary line is
an effective coupling, not an original rated transmission circuit.

In [ ]:
np.testing.assert_allclose(-detail['flow'],arrays['T_intact'],atol=1e-9)
payout=np.asarray(inputs['original_intact_duals'])@(-detail['flow'])
payout_error=float(np.max(np.abs(payout-np.asarray(inputs['original_intact_settlements']))))
print('Max intact-hour payout residual with ORIGINAL full-grid duals:',payout_error)
for row in rows:print(row)
print('Original constraints:',grid.E,'boundary off-diagonal couplings:',int(np.sum(np.triu(detail['reduced']<-1e-12,1))))
fig,ax=plt.subplots()
for key,label,color in [('terminal_relative_error','Correct sources: terminals','#88C0D0'),('flow_relative_error','Reconstructed full flows','#A3BE8C'),('omitted_source_relative_error','Source omission failure','#D08770')]:
    ax.semilogy([r['retained'] for r in rows],[max(r[key],1e-16) for r in rows],marker='o',label=label,color=color)
ax.set(xlabel='Retained bus count',ylabel='Relative error',title='Same 18 existing unit-transfer columns at every resolution');ax.legend();plt.show()

## A bounded resolution sweep, not a new injection ensemble
Start with the original six retained buses, then add other bus IDs in ascending
order to reach 12,18,26. This is a transparent diagnostic ordering, not optimized
zoning. Every case uses exactly the same saved H. At 26 buses no elimination
remains, so the source-omission control must coincide with the full solution.

The earlier 2,000-draw experiment compared regional angle averaging with Kron
and separately used random eliminated-bus injections. Its individual draws are
not retained here; the numbers below are a labeled replay of its saved summary.
Our fresh 18-path calculation has a different source-omission error and should
not be forced to match the old ensemble statistic.

In [ ]:
old=research['multiscale']
print('RETAINED SUMMARY ONLY:',{k:v for k,v in old.items() if k!='count_ensemble'})
fig,ax=plt.subplots()
labels=['Kron terminals','Regional averaging','Omitted sources','Corrected sources']
keys=['terminal_kron_relative_error','regional_spread_relative_error','omitted_internal_injection_error','corrected_internal_injection_error']
ax.bar(labels,[old[k] for k in keys],color=['#88C0D0','#B48EAD','#D08770','#A3BE8C'])
ax.set(ylabel='Relative angle-difference error',title='Earlier 2,000-draw experiment · summary replay only');plt.show()

## Count aggregation has its own compatible domain
The separate retained four-node illustration uses strengths [0.5,1,1.5,2],
cross-block Poisson event rates δsᵢsⱼ, δ=0.3, and positive-geometric jumps with
probability ρ=0.4. Summed intensity is δ(0.5+1)(1.5+2)=1.575, expected count
is intensity/ρ=3.9375, and P(count=0)=exp(−intensity).

Recompute those expectations analytically and compare the already saved 40,000
draw summary. No draws are regenerated. This checks compatibility of a prescribed
count model; it does not fit strengths, implement full multiscale inference,
or license applying integer-weight distributions to signed PTDFs.

In [ ]:
c=old['count_ensemble'];s=np.asarray(c['strengths']);intensity=c['delta']*s[:2].sum()*s[2:].sum()
assert np.isclose(intensity,c['coarse_intensity'])
assert np.isclose(intensity/c['rho'],c['expected_mean'])
assert np.isclose(np.exp(-intensity),c['expected_zero_probability'])
print(c)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].bar(['Analytic','Saved draws'],[intensity/c['rho'],c['observed_mean']],color=['#88C0D0','#B48EAD'])
axes[0].set(ylabel='Count mean',title='Compatible count domain only')
axes[1].bar(['Analytic','Saved draws'],[np.exp(-intensity),c['observed_zero_probability']],color=['#88C0D0','#B48EAD'])
axes[1].set(ylabel='Zero probability',title='No new samples drawn');fig.tight_layout();plt.show()

## What failed, and the next question
Omitting eliminated sources fails the boundary response. Replacing boundary
angles with regional means changes the target. Retaining terminal electrical
equivalence alone does not preserve internal feasible constraints, optimal
dispatch, shadow prices or congestion payouts. Reusing original duals checks
algebra, not a reduced-market pricing theorem.

Exercise: which original thermal inequalities would you need to carry into a
reduced dispatch problem? Why does agreement of terminal angles say nothing
about a count-network likelihood or hidden topology inferred from prices?

In [ ]:
bad=copy.deepcopy(inputs['requests']['6']);bad['inputs']['original_network']['weights'][0]*=2;reject('kron_response',bad)
bad=copy.deepcopy(inputs['requests']['6']);bad['inputs']['injections']['values'][0][0]+=1;reject('kron_response',bad)
checked=dict(resolution_checks=rows,intact_payout_error=payout_error,prior_monte_carlo='summary_only',new_synthetic_data=False,
    complete_18_original_paths=True,all_8638_intact_hours=True,original_full_grid_duals=True,reduced_market_equivalence=False)
(session.output/'consumer-summary.json').write_text(json.dumps(dict(checks=checked,detail={k:v.tolist() for k,v in detail.items()}),indent=2,allow_nan=False)+'\n')
summary=session.finish(checked,dict(operator_provenance=inputs['network_recovery']))
print(summary)